In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**Data Isolation & Dataset Subsetting**

In [ ]:
import os

BASE_PATH = '/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
TRAIN_SRC = os.path.join(BASE_PATH, 'train')
VALID_SRC = os.path.join(BASE_PATH, 'valid')
TEST_PATH = '/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/test/test'

# Create working folders for tomato subset
TOMATO_DIR = '/kaggle/working/tomato_dataset'
TRAIN_DIR = os.path.join(TOMATO_DIR, 'train')
VALID_DIR = os.path.join(TOMATO_DIR, 'valid')

def link_tomato_classes(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    for folder in os.listdir(src_dir):
        if folder.startswith('Tomato___'):
            src_folder = os.path.join(src_dir, folder)
            dst_folder = os.path.join(dst_dir, folder)
            if not os.path.exists(dst_folder):
                os.symlink(src_folder, dst_folder)
                print(f"Linked: {folder}")

link_tomato_classes(TRAIN_SRC, TRAIN_DIR)
link_tomato_classes(VALID_SRC, VALID_DIR)

**Data Loading & Pipeline Optimization**

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TOMATO_DIR = '/kaggle/working/tomato_dataset'

train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{TOMATO_DIR}/train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    f'{TOMATO_DIR}/valid',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print(f"\nLoaded {NUM_CLASSES} Tomato classes:")
for idx, name in enumerate(class_names):
    print(f"{idx}: {name}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)

**Model Architecture & Two-Stage Transfer Learning**

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1)
])

#5.3M Parameters
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet', # Pretrained weight learn from 1.4M image
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x) # 7 * 7 * 1280 = 62,720
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

# Stage 1: Feature Extraction
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',  # L = -sum_{i=1}^{C} y_i*log(hat{y}_i) || L = -log(hat{y}_c)
    metrics=['accuracy']
)

callbacks_stage1 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('best_tomato_stage1.keras', monitor='val_loss', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7)
]

print("Starting Stage 1 Training...")
history_stage1 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=8,
    callbacks=callbacks_stage1
)

# Stage 2: Fine-Tuning
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_stage2 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('tomato_efficientnet_finetuned.keras', monitor='val_loss', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7)
]

print("\nStarting Stage 2 Fine-Tuning...")
history_stage2 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=8,
    callbacks=callbacks_stage2
)

**Prediction And Result**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

TEST_PATH = '/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/test/test'
test_files = sorted([f for f in os.listdir(TEST_PATH) if f.startswith('Tomato')])

results = []

plt.figure(figsize=(15, 10))
for i, filename in enumerate(test_files):
    img_path = os.path.join(TEST_PATH, filename)
    raw_img = Image.open(img_path).resize((224, 224))
    
    img_array = tf.keras.utils.img_to_array(raw_img)
    img_batch = tf.expand_dims(img_array, 0)

    preds = model.predict(img_batch, verbose=0)[0]
    pred_idx = np.argmax(preds)
    pred_class = class_names[pred_idx].replace('Tomato___', '')
    confidence = preds[pred_idx]

    results.append({
        'Image': filename,
        'Predicted Disease': pred_class,
        'Confidence': f"{confidence:.2%}"
    })

    if i < 12:
        plt.subplot(3, 4, i + 1)
        plt.imshow(raw_img)
        plt.title(f"{filename}\nPred: {pred_class}\nConf: {confidence:.2%}", fontsize=8)
        plt.axis('off')

plt.tight_layout()
plt.show()

# Display prediction dataframe
results_df = pd.DataFrame(results)
results_df